# Xenium Pancreas 10x Data Audit and Optional SpatialData Conversion

This notebook checks the four downloaded 10x Xenium pancreas datasets before we start annotation and pseudotime.

Why this stage matters:

- the public 10x examples use different panels, so we need to know which genes are truly shared
- `spatialdata-io` can manage Xenium outputs nicely
- the downstream notebooks also keep a Scanpy fallback from `cell_feature_matrix.h5` and `cells.csv.gz`, which is faster for tabular annotation/modeling


In [ ]:

%matplotlib inline

import os
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/numba")

from pathlib import Path
import gc
import json
import importlib.util
import tarfile
import warnings
from io import StringIO

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 8
sns.set_style("white")

ROOT = Path("/Users/shihongwu/SpatioEv")
DATA_ROOT = Path("/Volumes/Shihong_5/for_spatioev/pancreas_Xenium_example_data_from_10X")
OUTPUT_DIR = ROOT / "data" / "xenium_pancreas_10x"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_CONFIGS = [
    {
        "sample_id": "pdac_pancreas_v1",
        "display_name": "Human Pancreas FFPE",
        "disease_group": "PDAC",
        "panel_hint": "Human Multi-Tissue and Cancer",
        "outs_path": DATA_ROOT / "Xenium_V1_human_Pancreas_FFPE_outs",
    },
    {
        "sample_id": "pdac_io_v1",
        "display_name": "Human Ductal Adenocarcinoma FFPE",
        "disease_group": "PDAC",
        "panel_hint": "Human Immuno-Oncology",
        "outs_path": DATA_ROOT / "Xenium_V1_Human_Ductal_Adenocarcinoma_FFPE_outs",
    },
    {
        "sample_id": "pdac_addon_v1",
        "display_name": "hPancreas Cancer Add-on FFPE",
        "disease_group": "PDAC",
        "panel_hint": "Human Multi-Tissue + Add-on",
        "outs_path": DATA_ROOT / "Xenium_V1_hPancreas_Cancer_Add_on_FFPE_outs",
    },
    {
        "sample_id": "normal_nondiseased_v1",
        "display_name": "hPancreas nondiseased section",
        "disease_group": "NormalPancreas",
        "panel_hint": "Human Multi-Tissue and Cancer",
        "outs_path": DATA_ROOT / "Xenium_V1_hPancreas_nondiseased_section_outs",
    },
]

def package_available(name):
    return importlib.util.find_spec(name) is not None

def save_df(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".pkl":
        df.to_pickle(path)
    else:
        df.to_csv(path, index=index)

def load_df(path):
    path = Path(path)
    if path.suffix == ".pkl":
        return pd.read_pickle(path)
    return pd.read_csv(path)

def present_columns(df, cols):
    return [c for c in cols if c in df.columns]

def make_sparse_safe_copy(X):
    return X.copy() if sparse.issparse(X) else np.asarray(X).copy()


## Package Status

SpatialData support is available when `spatialdata-io` is installed. The conversion cell below can write `.zarr` stores; if a future environment lacks SpatialData, the notebook records a clear install message and continues with Scanpy-compatible files.


In [ ]:

package_status = pd.Series(
    {
        "spatialdata": package_available("spatialdata"),
        "spatialdata_io": package_available("spatialdata_io"),
        "spatialdata_plot": package_available("spatialdata_plot"),
        "pyarrow": package_available("pyarrow"),
        "celltypist": package_available("celltypist"),
        "scvi": package_available("scvi"),
        "scanpy": package_available("scanpy"),
        "anndata": package_available("anndata"),
        "elpigraph": package_available("elpigraph"),
    },
    name="available",
).to_frame()
package_status


## Audit Raw Outputs

This reads only lightweight metadata plus the 10x feature matrix headers. It does not load transcript coordinates or full images.


In [ ]:

FOCUS_GENES = [
    "EPCAM", "KRT7", "KRT8", "KRT18", "KRT19", "SOX9", "MUC1", "MUC5AC",
    "TFF1", "TFF2", "TFF3", "CEACAM5", "CEACAM6", "AGR2", "AGR3", "S100P",
    "MKI67", "UBE2C", "TOP2A", "CENPF", "CDK1",
    "ACTA2", "PDGFRA", "FAP", "THY1", "PDPN", "DCN", "LUM", "VIM",
    "PECAM1", "VWF", "KDR", "CDH5",
    "PTPRC", "CD3D", "CD3E", "CD4", "CD8A", "FOXP3", "GZMB", "NKG7",
    "CD19", "MS4A1", "CD79A", "MZB1", "JCHAIN", "SDC1",
    "LST1", "LYZ", "CD68", "C1QA", "C1QB", "C1QC",
    "AMY2A", "PRSS1", "PRSS2", "CPA1", "CTRB1", "REG1A",
    "INS", "GCG", "SST", "PPY", "CHGA",
]

audit_rows = []
gene_sets = {}
focus_rows = []

for cfg in SAMPLE_CONFIGS:
    outs = Path(cfg["outs_path"])
    matrix_path = outs / "cell_feature_matrix.h5"
    cells_path = outs / "cells.csv.gz"
    metrics_path = outs / "metrics_summary.csv"
    gene_panel_path = outs / "gene_panel.json"
    cell_groups_path = outs / "cell_groups.csv"

    missing = [
        name
        for name, path in {
            "cell_feature_matrix.h5": matrix_path,
            "cells.csv.gz": cells_path,
            "metrics_summary.csv": metrics_path,
            "gene_panel.json": gene_panel_path,
        }.items()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(f"{cfg['sample_id']} missing required files: {missing}")

    adata_head = sc.read_10x_h5(matrix_path)
    gene_sets[cfg["sample_id"]] = set(adata_head.var_names)
    cells_head = pd.read_csv(cells_path, nrows=5)
    metrics = pd.read_csv(metrics_path)
    metrics_row = metrics.iloc[0].to_dict() if len(metrics) else {}

    audit_rows.append(
        {
            "sample_id": cfg["sample_id"],
            "display_name": cfg["display_name"],
            "disease_group": cfg["disease_group"],
            "panel_hint": cfg["panel_hint"],
            "outs_path": str(outs),
            "n_cells_matrix": int(adata_head.n_obs),
            "n_genes": int(adata_head.n_vars),
            "cells_csv_columns": ", ".join(cells_head.columns),
            "has_cell_groups_csv": cell_groups_path.exists(),
            "panel_name_metrics": metrics_row.get("panel_name", np.nan),
            "num_cells_metrics": metrics_row.get("num_cells_detected", np.nan),
            "median_genes_per_cell": metrics_row.get("median_genes_per_cell", np.nan),
            "median_transcripts_per_cell": metrics_row.get("median_transcripts_per_cell", np.nan),
        }
    )

    for gene in FOCUS_GENES:
        focus_rows.append(
            {
                "sample_id": cfg["sample_id"],
                "disease_group": cfg["disease_group"],
                "gene": gene,
                "present": gene in adata_head.var_names,
            }
        )

audit_df = pd.DataFrame(audit_rows)
focus_gene_df = pd.DataFrame(focus_rows)

common_genes = sorted(set.intersection(*gene_sets.values()))
union_genes = sorted(set.union(*gene_sets.values()))

save_df(audit_df, OUTPUT_DIR / "xenium_dataset_audit.csv")
save_df(focus_gene_df, OUTPUT_DIR / "xenium_focus_gene_availability.csv")
pd.Series(common_genes, name="gene").to_csv(OUTPUT_DIR / "xenium_common_genes.csv", index=False)

print(f"Common genes across all four datasets: {len(common_genes)}")
print(f"Union genes across all four datasets: {len(union_genes)}")
audit_df


In [ ]:

focus_matrix = (
    focus_gene_df
    .pivot_table(index="gene", columns="sample_id", values="present", aggfunc="first")
    .reindex(FOCUS_GENES)
)

plt.figure(figsize=(8, 10))
sns.heatmap(
    focus_matrix.astype(float),
    cmap=sns.color_palette(["#f2f2f2", "#2b8cbe"], as_cmap=True),
    cbar=False,
    linewidths=0.2,
    linecolor="white",
)
plt.title("Focus gene availability across Xenium panels")
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()
plt.show()


## Optional: Convert Raw Xenium Outputs to SpatialData Zarr

The SpatialData docs describe Xenium loading through `spatialdata_io.xenium(path, ...)`, which reads the Xenium `outs` folder into a `SpatialData` object containing images, labels, points, shapes, and an AnnData table. This conversion is skipped automatically unless `spatialdata_io` is installed.


In [ ]:

SPATIALDATA_DIR = OUTPUT_DIR / "spatialdata_zarr"
SPATIALDATA_DIR.mkdir(exist_ok=True)
FORCE_SPATIALDATA_CONVERT = False

if not package_available("spatialdata_io"):
    print(
        "spatialdata_io is not installed in this environment. "
        "If you want SpatialData conversion, install spatialdata, spatialdata-io, spatialdata-plot, and pyarrow in spatioev_env, then rerun this cell."
    )
else:
    from spatialdata_io import xenium

    for cfg in SAMPLE_CONFIGS:
        zarr_path = SPATIALDATA_DIR / f"{cfg['sample_id']}.zarr"
        if zarr_path.exists() and not FORCE_SPATIALDATA_CONVERT:
            print(f"Already exists: {zarr_path}")
            continue

        print(f"Converting {cfg['sample_id']} to SpatialData zarr...")
        sdata = xenium(
            cfg["outs_path"],
            cells_boundaries=False,
            nucleus_boundaries=False,
            cells_as_circles=True,
            transcripts=False,
            cells_labels=False,
            nucleus_labels=False,
            morphology_mip=False,
            morphology_focus=False,
            aligned_images=False,
        )
        sdata.write(zarr_path)
        print(f"Wrote {zarr_path}")
